# 02b — Gather pool-level APY + TVL history

## What this notebook does

Notebook 02 fetches **protocol-level** TVL (one number per protocol per day).

This notebook fetches **pool-level** APY and TVL history from DeFiLlama's yields API.
For each pool listed in `pool_mapping.csv` we call:

- `https://yields.llama.fi/chart/{pool_id}` → daily APY (base + reward) and TVL in USD

We also re-pull the current snapshot from:

- `https://yields.llama.fi/pools` → current APY/TVL for every pool we care about

Outputs:

- `data/processed/pool_yields_timeseries.csv` — one row per (date, pool)
- `data/processed/pool_yields_summary.csv` — one row per pool with current state
- `data/raw/yields_pools_snapshot.csv` — full snapshot of the matched pools (raw)

This is the main dataset we will use in notebooks 03 (explore), 04 (model), 05 (communicate).

In [ ]:
import time
import requests
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## Load the pool mapping from notebook 01

In [ ]:
pool_mapping_df = pd.read_csv(DATA_PROCESSED / "pool_mapping.csv")
print(f"Pools to fetch: {len(pool_mapping_df)}")
pool_mapping_df.head()

## Step A — current snapshot of all our pools

Pull the full `/pools` endpoint once and filter down to the pool IDs we want. We keep the raw matched slice in `data/raw/`.

In [ ]:
POOLS_URL = "https://yields.llama.fi/pools"

resp = requests.get(POOLS_URL, timeout=60)
resp.raise_for_status()
all_pools = pd.DataFrame(resp.json()["data"])
print(f"Total pools in DeFiLlama: {len(all_pools):,}")

snapshot = all_pools[all_pools["pool"].isin(pool_mapping_df["pool_id"])].copy()
# Only pull in the columns the API does not already provide (avoids project/chain/symbol collisions).
snapshot = snapshot.merge(
    pool_mapping_df[["pool_id", "category", "asset"]],
    left_on="pool", right_on="pool_id", how="left",
)
print(f"Matched pools: {len(snapshot)} / {len(pool_mapping_df)}")

snapshot.to_csv(DATA_RAW / "yields_pools_snapshot.csv", index=False)
snapshot[["category", "project", "chain", "asset", "symbol", "tvlUsd", "apy", "apyBase", "apyReward"]]

## Step B — historical APY + TVL per pool

For each pool we call `/chart/{pool_id}` and normalize the response into a tidy dataframe.

In [ ]:
CHART_URL = "https://yields.llama.fi/chart/{pool_id}"

def fetch_pool_chart(pool_id: str) -> pd.DataFrame:
    """Return a tidy DataFrame of date / apy / apyBase / apyReward / tvl_usd for one pool."""
    r = requests.get(CHART_URL.format(pool_id=pool_id), timeout=60)
    r.raise_for_status()
    payload = r.json()
    if payload.get("status") != "success":
        return pd.DataFrame()
    df = pd.DataFrame(payload["data"])
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["timestamp"], errors="coerce")
    out = df[["date", "apy", "apyBase", "apyReward", "tvlUsd"]].rename(
        columns={"tvlUsd": "tvl_usd"}
    )
    for c in ["apy", "apyBase", "apyReward", "tvl_usd"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return out

In [ ]:
frames = []
for _, row in pool_mapping_df.iterrows():
    pool_id = row["pool_id"]
    try:
        hist = fetch_pool_chart(pool_id)
    except Exception as exc:
        print(f"  [skip] {row['project']:14s} {row['chain']:10s} {row['symbol']:6s}  ({exc})")
        continue
    if hist.empty:
        print(f"  [empty] {row['project']:14s} {row['chain']:10s} {row['symbol']:6s}")
        continue
    for col in ["pool_id", "category", "project", "chain", "asset", "symbol"]:
        hist[col] = row[col]
    frames.append(hist)
    print(f"  {row['project']:14s} {row['chain']:10s} {row['symbol']:6s}  rows={len(hist):4d}  from {hist['date'].min().date()} to {hist['date'].max().date()}")
    time.sleep(0.2)  # gentle on the API

pool_yields_timeseries_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal rows: {len(pool_yields_timeseries_df):,}")
pool_yields_timeseries_df.head()

## Step C — current summary table

One row per pool with: current APY, current TVL, average APY over the last 30 / 90 / 365 days, APY volatility.

In [ ]:
def summarize_pool(group: pd.DataFrame) -> pd.Series:
    group = group.sort_values("date")
    latest = group.iloc[-1]
    last_30  = group[group["date"] >= group["date"].max() - pd.Timedelta(days=30)]
    last_90  = group[group["date"] >= group["date"].max() - pd.Timedelta(days=90)]
    last_365 = group[group["date"] >= group["date"].max() - pd.Timedelta(days=365)]
    return pd.Series({
        "first_date":       group["date"].min(),
        "last_date":        group["date"].max(),
        "n_days":           len(group),
        "current_tvl_usd":  latest["tvl_usd"],
        "current_apy":      latest["apy"],
        "apy_mean_30d":     last_30["apy"].mean(),
        "apy_mean_90d":     last_90["apy"].mean(),
        "apy_mean_365d":    last_365["apy"].mean(),
        "apy_std_365d":     last_365["apy"].std(),
    })

key_cols = ["category", "project", "chain", "asset", "symbol", "pool_id"]
pool_yields_summary_df = (
    pool_yields_timeseries_df
    .groupby(key_cols, as_index=False)
    .apply(summarize_pool)
    .reset_index(drop=True)
)
pool_yields_summary_df = pool_yields_summary_df.sort_values(["category", "chain", "asset"]).reset_index(drop=True)
pool_yields_summary_df

## Save the datasets

In [ ]:
timeseries_path = DATA_PROCESSED / "pool_yields_timeseries.csv"
summary_path    = DATA_PROCESSED / "pool_yields_summary.csv"

pool_yields_timeseries_df.to_csv(timeseries_path, index=False)
pool_yields_summary_df.to_csv(summary_path, index=False)

print("Saved:")
print(" ", timeseries_path, f"({len(pool_yields_timeseries_df):,} rows)")
print(" ", summary_path,    f"({len(pool_yields_summary_df):,} rows)")

## What we have now

After this notebook:

- A clean pool-level **time-series** dataset with daily APY + TVL across staking / lending / stablecoin yield pools, on Ethereum / Arbitrum / Base, for ETH / USDC / USDT / DAI / USDe / USDS.
- A pool-level **summary** table for quick cross-sectional comparison.

Next notebooks:

- **03_explore.ipynb** — summary statistics, distribution plots, time-series plots per category.
- **04_external_factors.ipynb** — ETH price (CoinGecko), Fed funds rate (FRED), staked-ETH ratio (beaconcha.in).
- **05_model.ipynb** — regressions for the three research questions:
  - Q1: `staking_APY ~ staked_ETH_share + gas_fees`
  - Q2: `stablecoin_APY ~ FedFunds`
  - Q3: `log(TVL) ~ APY + category + chain`